# Solutions · 10 · The linear-quadratic regulator

Worked solutions to the exercises of [notebook 10](../notebooks/10_the_linear_quadratic_regulator.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, statespace as sp, linalg as la, response as rs, design as dg, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Apply Bryson's rule to the cart-pendulum: allowed excursions 0.1 m, 0.5 m/s, 0.05 rad, 0.5 rad/s and 5 N. Simulate the recovery from 10 degrees and check whether the force stays within 5 N; adjust R until it does.

In [2]:
cp = models.cart_pendulum()["system"]
A, B, C = cp.A, cp.B, cp.C
Q = np.diag(1 / np.array([0.1, 0.5, 0.05, 0.5]) ** 2)                 # Bryson: 1 / (allowed excursion)^2
R0 = 1 / 5.0**2
for R in (R0, 4 * R0, 16 * R0, 64 * R0):
    r = sp.lqr(A, B, Q, np.array([[R]]))
    Acl = sy.ss(A - B @ r["K"], B, C, np.zeros((2, 1)))
    t, y, x = rs.initial(Acl, [0, 0, np.radians(10), 0], 5.0, 5001)
    u = -(x @ r["K"].T)[:, 0]
    i_settle = min(np.max(np.flatnonzero(np.abs(x[:, 2]) > 0.002)) + 1, len(t) - 1)
    print(f"R = {R / R0:.0f} x Bryson: peak force {np.max(np.abs(u)):.2f} N, cart excursion {100 * np.max(np.abs(x[:, 0])):.1f} cm, angle settled in {t[i_settle]:.2f} s" + (" (not within 5 s)" if i_settle == len(t) - 1 else ""))

R = 1 x Bryson: peak force 30.83 N, cart excursion 11.4 cm, angle settled in 2.47 s
R = 4 x Bryson: peak force 16.74 N, cart excursion 12.5 cm, angle settled in 2.54 s
R = 16 x Bryson: peak force 9.70 N, cart excursion 14.3 cm, angle settled in 2.67 s
R = 64 x Bryson: peak force 6.20 N, cart excursion 17.1 cm, angle settled in 2.91 s


Bryson's rule gives a first weighting from the allowed excursions; the 10° recovery it produces asks for
30 N, six times the budget, and raising R - cheaper to tell the regulator that force is expensive than to
re-weight every state - brings the peak force down by about half per factor of four, under 5 N at 64 times
Bryson's R, at the cost of a larger cart excursion and a slower recovery. Three or four such iterations are
the normal LQR design loop; the rule is a starting point, not a result.

## Exercise 2

For the DC motor speed loop compare LQR with integral action (qi = 10, 100, 1000) against a PI controller tuned for the same settling time: overshoot, control effort, and response to a load torque step (an input disturbance).

In [3]:
mot = models.dc_motor()["system"]
A, B, C = mot.A, mot.B, mot.C
def recovery(td, yd):
    out = np.flatnonzero(np.abs(yd) > 0.02 * np.max(np.abs(yd)))
    return td[min(np.max(out) + 1, len(td) - 1)]
rows = []
for qi in (100.0, 1000.0, 10000.0):
    li = sp.lqr_integral(A, B, C, np.diag([1.0, 10.0]), np.array([[0.01]]), qi=qi)
    Aa, Ba = li["Aa"], li["Ba"]; Ka = np.hstack([li["K"], li["Ki"]])
    Acl = Aa - Ba @ Ka
    cl_r = sy.ss(Acl, np.array([[0.0], [0.0], [1.0]]), np.hstack([C, [[0.0]]]), 0.0)         # reference -> speed
    cl_d = sy.ss(Acl, np.vstack([B, [[0.0]]]), np.hstack([C, [[0.0]]]), 0.0)               # a disturbance at the plant input (voltage)
    t, y = rs.step(cl_r, 4.0); td, yd = rs.step(cl_d, 4.0)
    info = rs.step_info(t, y, final=1.0)
    rows.append((f"LQR+I qi={qi:.0f}", info["overshoot"], info["settling_time"], np.max(np.abs(yd)), recovery(td, yd)))
# a PI tuned by SIMC on the FOPDT fit of the motor
t_, y_ = rs.step(mot, 3.0, 3001)
fo = dg.fopdt_from_step(t_, y_)
pi = dg.imc_pid(fo["K"], fo["tau"], max(fo["L"], 0.02), lam=0.1)
cl = dg.closed_loop(mot.to_tf(), dg.pid_standard(pi["Kp"], pi["Ti"]))
t, y = rs.step(cl["T"], 4.0); info = rs.step_info(t, y, final=1.0)
td, yd = rs.step(cl["PS"], 4.0)                                                           # the same input disturbance, through PS
rows.append(("PI (SIMC, lambda = 0.1)", info["overshoot"], info["settling_time"], np.max(np.abs(yd)), recovery(td, yd)))
print(pd.DataFrame(rows, columns=["controller", "overshoot %", "settling (s)", "peak speed deviation (per V of disturbance)", "disturbance recovered (s)"]).to_string(index=False, float_format="%.3f"))

             controller  overshoot %  settling (s)  peak speed deviation (per V of disturbance)  disturbance recovered (s)
           LQR+I qi=100        0.000           NaN                                        0.007                      4.000
          LQR+I qi=1000        0.000         1.404                                        0.006                      1.588
         LQR+I qi=10000        0.576         0.488                                        0.004                      0.600
PI (SIMC, lambda = 0.1)        4.536         0.796                                        0.025                      2.428


Raising the integral weight speeds up the rejection of an input disturbance and the removal of the offset,
with a growing overshoot on the setpoint - the same trade-off as a PI's integral time, but with the state
feedback on current and speed providing damping that a PI must obtain from its proportional gain alone. The
LQR-with-integral loop holds the speed deviation smaller and recovers sooner than the PI of comparable
settling time; it also uses the current measurement, which the PI does not.

## Exercise 3

**Implement it yourself:** compute the LQR cost J = int (x'Qx + u'Ru) dt by simulating the closed loop from x0 with `response.initial` and integrating numerically; compare with x0' P x0 and show that any other stabilising gain (e.g. from pole placement) gives a larger cost.

In [4]:
cp = models.cart_pendulum()["system"]
A, B, C = cp.A, cp.B, cp.C
Q, R = np.diag([10.0, 1.0, 100.0, 1.0]), np.array([[0.1]])
x0 = np.array([0.0, 0.0, 0.2, 0.0])
r = sp.lqr(A, B, Q, R)
def cost(K):
    Acl = sy.ss(A - B @ K, B, C, np.zeros((2, 1)))
    t, y, x = rs.initial(Acl, x0, 20.0, 20001)
    u = -(x @ K.T)
    integrand = np.einsum("ij,jk,ik->i", x, Q, x) + np.einsum("ij,jk,ik->i", u, R, u)
    return np.sum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(t))
print(f"LQR gain: simulated cost {cost(r['K']):.4f} vs x0' P x0 = {sp.cost_to_go(r['P'], x0):.4f}")
for des in ([-2 + 2j, -2 - 2j, -5.0, -6.0], [-1 + 1j, -1 - 1j, -3.0, -4.0], [-4 + 4j, -4 - 4j, -10.0, -12.0]):
    Kp_ = sp.place(A, B, des)
    print(f"pole placement at {np.round(des, 1)}: cost {cost(Kp_):.4f} (larger, as optimality requires)")

LQR gain: simulated cost 1.3393 vs x0' P x0 = 1.3393


pole placement at [-2.+2.j -2.-2.j -5.+0.j -6.+0.j]: cost 1.7478 (larger, as optimality requires)


pole placement at [-1.+1.j -1.-1.j -3.+0.j -4.+0.j]: cost 5.1142 (larger, as optimality requires)
pole placement at [ -4.+4.j  -4.-4.j -10.+0.j -12.+0.j]: cost 2.8748 (larger, as optimality requires)


The simulated cost of the LQR gain equals x₀ᵀPx₀ to the quadrature's accuracy, and every placed-pole gain
costs more - slower ones through a large state integral, faster ones through control effort. The Riccati
solution is the minimum over *all* stabilising gains, which is why it is called optimal.